# 🎵 Lyrics Search Tool - Multi-Source Edition

Search and retrieve lyrics for any song using **multiple sources**.

## Features:

✅ **Multiple sources** - Tries different APIs automatically
✅ **No API key needed** - Works out of the box
✅ **Batch search** - Process multiple songs at once
✅ **Export options** - Save as TXT, JSON, or CSV

---

**Examples:**
- The Animals - House of the Rising Sun
- Rolê da Diagrama - Piseiro do Baião
- Taylor Swift - Anti-Hero


## 1️⃣ Setup & Installation

In [ ]:
# Install required packages
print("Installing dependencies...")
!pip install -q requests beautifulsoup4 lxml
print("✓ Installation complete!")

## 2️⃣ Lyrics Fetcher Class

This class tries multiple sources to find lyrics.

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import time
from urllib.parse import quote, quote_plus
import json

class LyricsFetcher:
    """Fetch lyrics from multiple sources"""
    
    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        })
    
    def clean_string(self, s):
        """Clean string for URL"""
        s = s.lower()
        s = re.sub(r'[^a-z0-9\s]', '', s)
        s = re.sub(r'\s+', '-', s.strip())
        return s
    
    def search_lyrics_ovh(self, artist, title):
        """Try lyrics.ovh API (free, no key needed)"""
        try:
            url = f"https://api.lyrics.ovh/v1/{quote(artist)}/{quote(title)}"
            response = self.session.get(url, timeout=10)
            
            if response.status_code == 200:
                data = response.json()
                if 'lyrics' in data and data['lyrics']:
                    return {
                        'lyrics': data['lyrics'].strip(),
                        'source': 'lyrics.ovh',
                        'artist': artist,
                        'title': title
                    }
        except Exception as e:
            pass
        return None
    
    def search_azlyrics(self, artist, title):
        """Try AZLyrics (web scraping)"""
        try:
            # Clean artist and title for URL
            clean_artist = self.clean_string(artist)
            clean_title = self.clean_string(title)
            
            url = f"https://www.azlyrics.com/lyrics/{clean_artist}/{clean_title}.html"
            
            response = self.session.get(url, timeout=10)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                
                # Find lyrics div (AZLyrics specific)
                lyrics_divs = soup.find_all('div', class_=None, id=None)
                for div in lyrics_divs:
                    if div.get('class') is None and div.get('id') is None:
                        text = div.get_text().strip()
                        if len(text) > 100 and '\n' in text:
                            return {
                                'lyrics': text,
                                'source': 'AZLyrics',
                                'artist': artist,
                                'title': title,
                                'url': url
                            }
        except Exception as e:
            pass
        return None
    
    def search_letrasmusbr(self, artist, title):
        """Try Letras.mus.br (good for Brazilian music)"""
        try:
            clean_artist = self.clean_string(artist)
            clean_title = self.clean_string(title)
            
            url = f"https://www.letras.mus.br/{clean_artist}/{clean_title}/"
            
            response = self.session.get(url, timeout=10)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                
                # Find lyrics container
                lyrics_div = soup.find('div', class_='lyric-original')
                if not lyrics_div:
                    lyrics_div = soup.find('div', class_='cnt-letra')
                
                if lyrics_div:
                    lyrics = lyrics_div.get_text().strip()
                    if len(lyrics) > 50:
                        return {
                            'lyrics': lyrics,
                            'source': 'Letras.mus.br',
                            'artist': artist,
                            'title': title,
                            'url': url
                        }
        except Exception as e:
            pass
        return None
    
    def search_vagalume(self, artist, title):
        """Try Vagalume (Brazilian lyrics site)"""
        try:
            clean_artist = self.clean_string(artist)
            clean_title = self.clean_string(title)
            
            url = f"https://www.vagalume.com.br/{clean_artist}/{clean_title}.html"
            
            response = self.session.get(url, timeout=10)
            
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')
                
                # Find lyrics container
                lyrics_div = soup.find('div', id='lyrics')
                
                if lyrics_div:
                    lyrics = lyrics_div.get_text().strip()
                    if len(lyrics) > 50:
                        return {
                            'lyrics': lyrics,
                            'source': 'Vagalume',
                            'artist': artist,
                            'title': title,
                            'url': url
                        }
        except Exception as e:
            pass
        return None
    
    def search(self, artist, title, verbose=True):
        """Search all sources in order"""
        if verbose:
            print(f"🔍 Searching: {title} by {artist}")
        
        # Try each source
        sources = [
            ('lyrics.ovh', self.search_lyrics_ovh),
            ('Letras.mus.br', self.search_letrasmusbr),
            ('Vagalume', self.search_vagalume),
            ('AZLyrics', self.search_azlyrics),
        ]
        
        for source_name, search_func in sources:
            if verbose:
                print(f"  Trying {source_name}...", end=" ")
            
            result = search_func(artist, title)
            
            if result:
                if verbose:
                    print("✓ Found!")
                return result
            else:
                if verbose:
                    print("✗")
            
            # Small delay between attempts
            time.sleep(0.5)
        
        if verbose:
            print("  ❌ Not found in any source")
        return None

# Initialize fetcher
fetcher = LyricsFetcher()
print("✓ Lyrics fetcher initialized!")

## 3️⃣ Single Song Search

In [ ]:
# Enter song details
artist_name = "The Animals"
song_title = "House of the Rising Sun"

print("\n" + "="*80)
result = fetcher.search(artist_name, song_title)
print("="*80)

if result:
    print(f"\n🎵 {result['title']}")
    print(f"👤 {result['artist']}")
    print(f"📍 Source: {result['source']}")
    if 'url' in result:
        print(f"🔗 {result['url']}")
    print("\n" + "-"*80)
    print(result['lyrics'])
    print("-"*80)
else:
    print("\n❌ Could not find lyrics for this song.")
    print("\nTips:")
    print("  • Check spelling of artist and song name")
    print("  • Try without special characters")
    print("  • Try a different version (acoustic, live, etc.)")

## 4️⃣ Batch Search - Multiple Songs

In [ ]:
# List of songs (format: "Artist - Song Title")
songs_to_search = [
    "The Animals - House of the Rising Sun",
    "Rolê da Diagrama - Piseiro do Baião",
    "Taylor Swift - Anti-Hero",
    # Add more songs here
]

print(f"\n{'='*80}")
print(f" Searching {len(songs_to_search)} songs ".center(80, "="))
print(f"{'='*80}\n")

results = []

for i, song_string in enumerate(songs_to_search, 1):
    print(f"\n[{i}/{len(songs_to_search)}] {song_string}")
    print("-" * 80)
    
    try:
        # Parse the string
        if " - " in song_string:
            artist, title = song_string.split(" - ", 1)
            artist = artist.strip()
            title = title.strip()
        else:
            print(f"⚠️  Invalid format (use 'Artist - Title')")
            continue
        
        result = fetcher.search(artist, title)
        
        if result:
            results.append(result)
            print(f"\n✓ Success! Found via {result['source']}")
        else:
            print(f"\n✗ Failed - lyrics not found")
        
        # Delay between songs to be respectful
        if i < len(songs_to_search):
            time.sleep(1)
            
    except Exception as e:
        print(f"  ❌ Error: {e}")

print(f"\n\n{'='*80}")
print(f" Results: {len(results)}/{len(songs_to_search)} songs found ".center(80, "="))
print(f"{'='*80}\n")

## 5️⃣ Display All Lyrics

In [ ]:
# Display all found lyrics
if results:
    for i, song in enumerate(results, 1):
        print("\n\n")
        print("#" * 80)
        print(f"## Song {i}/{len(results)}")
        print("#" * 80)
        print(f"\n🎵 {song['title']}")
        print(f"👤 {song['artist']}")
        print(f"📍 Source: {song['source']}")
        if 'url' in song:
            print(f"🔗 {song['url']}")
        print("\n" + "-" * 80)
        print(song['lyrics'])
        print("-" * 80)
else:
    print("No lyrics found yet. Run the search cells above first!")

## 6️⃣ Save to Text Files

In [ ]:
import os

# Create output directory
output_dir = "lyrics_output"
os.makedirs(output_dir, exist_ok=True)

def sanitize_filename(name):
    """Remove invalid characters from filename"""
    return re.sub(r'[<>:"/\\|?*]', '', name)

if results:
    print("Saving lyrics to files...\n")
    
    for song in results:
        filename = f"{sanitize_filename(song['artist'])} - {sanitize_filename(song['title'])}.txt"
        filepath = os.path.join(output_dir, filename)
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(f"{song['title']}\n")
            f.write(f"by {song['artist']}\n")
            f.write(f"Source: {song['source']}\n")
            if 'url' in song:
                f.write(f"URL: {song['url']}\n")
            f.write("\n" + "="*60 + "\n\n")
            f.write(song['lyrics'])
        
        print(f"✓ {filename}")
    
    print(f"\n✓ All lyrics saved to /{output_dir}/")
else:
    print("No lyrics to save. Run the search cells first!")

## 7️⃣ Export to JSON

In [ ]:
# Save as JSON
if results:
    json_file = os.path.join(output_dir, 'lyrics_collection.json')
    
    with open(json_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    
    print(f"✓ JSON saved: {json_file}")
    print(f"\nContains {len(results)} songs with lyrics")
else:
    print("No lyrics to export. Run the search cells first!")

## 8️⃣ Export to CSV

In [ ]:
import csv

# Save as CSV
if results:
    csv_file = os.path.join(output_dir, 'lyrics_collection.csv')
    
    with open(csv_file, 'w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['artist', 'title', 'source', 'url', 'lyrics'])
        writer.writeheader()
        
        for song in results:
            writer.writerow({
                'artist': song['artist'],
                'title': song['title'],
                'source': song['source'],
                'url': song.get('url', ''),
                'lyrics': song['lyrics']
            })
    
    print(f"✓ CSV saved: {csv_file}")
else:
    print("No lyrics to export. Run the search cells first!")

## 9️⃣ Download All Files (Google Colab)

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB and results:
    import shutil
    from google.colab import files
    
    # Create ZIP archive
    print("Creating ZIP archive...")
    zip_file = shutil.make_archive('lyrics_collection', 'zip', output_dir)
    
    # Download
    print("Starting download...")
    files.download(zip_file)
    print("✓ Download complete!")
elif not IN_COLAB:
    print(f"Not running in Colab. Files are saved in: {os.path.abspath(output_dir)}")
else:
    print("No files to download. Run the search cells first!")

## 🔟 Quick Search Widget (Interactive)

Use this widget for quick one-off searches!

In [ ]:
try:
    from ipywidgets import widgets, Layout
    from IPython.display import display, clear_output, HTML
    
    # Create input widgets
    artist_input = widgets.Text(
        placeholder='Enter artist name',
        description='Artist:',
        layout=Layout(width='60%')
    )
    
    song_input = widgets.Text(
        placeholder='Enter song title',
        description='Song:',
        layout=Layout(width='60%')
    )
    
    search_button = widgets.Button(
        description='🔍 Search Lyrics',
        button_style='success',
        layout=Layout(width='200px', height='40px')
    )
    
    output = widgets.Output()
    
    def on_search_clicked(b):
        with output:
            clear_output()
            
            artist = artist_input.value.strip()
            title = song_input.value.strip()
            
            if not artist or not title:
                print("⚠️  Please enter both artist and song name")
                return
            
            print("="*80)
            result = fetcher.search(artist, title)
            print("="*80)
            
            if result:
                print(f"\n🎵 {result['title']}")
                print(f"👤 {result['artist']}")
                print(f"📍 Source: {result['source']}")
                if 'url' in result:
                    print(f"🔗 {result['url']}")
                print("\n" + "-"*80)
                print(result['lyrics'])
                print("-"*80)
            else:
                print("\n❌ Could not find lyrics.")
                print("\nTips:")
                print("  • Check spelling")
                print("  • Try without special characters")
                print("  • Try a simpler song title")
    
    search_button.on_click(on_search_clicked)
    
    # Display the widget
    display(HTML("<h3>🎵 Quick Lyrics Search</h3>"))
    display(widgets.VBox([
        artist_input,
        song_input,
        search_button,
        output
    ]))
    
except ImportError:
    print("Widget support not available. Use the search cells above instead.")

## 📋 Summary & Tips

### What This Notebook Does:

✅ Searches multiple lyrics sources automatically
✅ Works without API keys
✅ Handles Brazilian and international music
✅ Exports in multiple formats (TXT, JSON, CSV)

### Sources Used:

1. **lyrics.ovh** - Free API, good for international songs
2. **Letras.mus.br** - Brazilian lyrics (excellent for MPB, Sertanejo, Funk, etc.)
3. **Vagalume** - Another Brazilian source
4. **AZLyrics** - Large international database

### Tips for Better Results:

🎯 **Use correct spelling** - Artist and song names matter
🎯 **Try variations** - "The Beatles" vs "Beatles"
🎯 **Keep it simple** - Avoid extra text like "(Official)"
🎯 **For Brazilian songs** - Sources are optimized for PT-BR

### Common Issues:

❓ **Song not found?**
- Check spelling
- Try without accents/special characters
- Some very new or obscure songs may not be available

❓ **Rate limits?**
- The notebook includes delays between requests
- If blocked, wait a few minutes and try again

### Legal Note:

⚖️ Lyrics are copyrighted. Use for:
- Personal study and analysis
- Educational purposes
- Research projects

Do not redistribute commercially.

---

**Enjoy your music research! 🎵**